# Deploy AgentCore Memory for Betting & Gaming Demo

Creates an AgentCore Memory resource with three long-term strategies:
- **Semantic** — extracts facts (e.g. "John placed a bet on Dortmund")
- **Summary** — generates session summaries
- **User Preference** — captures preferences (e.g. "prefers football betting")

Short-term memory is included by default for within-session conversation persistence.

<hr>

In [ ]:
!pip install bedrock-agentcore --quiet

## Step 1: Create Memory Resource\n
<hr>

In [13]:
from bedrock_agentcore.memory import MemoryClient
import time

REGION = "us-east-1"
# Set this to an existing memory ID to skip creation, or leave as None to create a new one
memory_id = "agentic_ai_demo-U5VYKlHuUT"

STRATEGIES = [
    {
        "semanticMemoryStrategy": {
            "name": "FactExtractor",
            "namespaceTemplates": ["/facts/actor/{actorId}/session/{sessionId}/"]
        }
    },
    {
        "summaryMemoryStrategy": {
            "name": "SessionSummarizer",
            "namespaceTemplates": ["/summaries/actor/{actorId}/session/{sessionId}/"]
        }
    },
    {
        "userPreferenceMemoryStrategy": {
            "name": "PreferenceLearner",
            "namespaceTemplates": ["/preferences/actor/{actorId}/"]
        }
    },
]

client = MemoryClient(region_name=REGION)
boto_client = client.gmcp_client

# 1. If a memory_id is provided, verify it exists
if memory_id:
    try:
        resp = boto_client.get_memory(memoryId=memory_id)
        status = resp.get('memory', {}).get('status', 'UNKNOWN')
        print(f"Memory already exists: {memory_id} (status: {status})")
    except Exception as e:
        print(f"Memory ID '{memory_id}' not found: {e}")
        memory_id = None

# 2. If no valid memory_id, create a new memory
if not memory_id:
    print("Creating new memory...")
    resp = boto_client.create_memory(
        name="agentic_ai_demo",
        description="Agentic AI Demo Memory",
        eventExpiryDuration=90,
        memoryStrategies=STRATEGIES
    )
    memory_id = resp["memory"]["id"]
    print(f"Created memory: {memory_id}, waiting for ACTIVE...")

# 3. Wait for ACTIVE status
for _ in range(60):
    info = boto_client.get_memory(memoryId=memory_id)
    st = info["memory"]["status"]
    if st == "ACTIVE":
        print("Memory is ACTIVE.")
        break
    elif st in ("FAILED", "DELETING"):
        raise RuntimeError(f"Memory creation failed: {st}")
    time.sleep(5)
else:
    print("Warning: timed out waiting for ACTIVE.")

print(f"\nMemory ID: {memory_id}")
print("Save this ID — you need it in the agent code.")

Memory already exists: agentic_ai_demo-U5VYKlHuUT (status: ACTIVE)
Memory is ACTIVE.

Memory ID: agentic_ai_demo-U5VYKlHuUT
Save this ID — you need it in the agent code.


## Step 2: Verify Memory Resource\n
<hr>

In [14]:
# Verify the memory resource
resp = boto_client.get_memory(memoryId=memory_id)
mem = resp["memory"]
print(f"Memory ID: {mem.get('id')}")
print(f"Status: {mem.get('status')}")

strategies = mem.get('strategies', [])
print(f"\nStrategies ({len(strategies)}):")
for s in strategies:
    print(f"  - {s.get('name')} (id: {s.get('strategyId', 'unknown')})")

Memory ID: agentic_ai_demo-U5VYKlHuUT
Status: ACTIVE

Strategies (3):
  - FactExtractor (id: FactExtractor-jWO2ymAkuA)
  - PreferenceLearner (id: PreferenceLearner-lQpTW43hye)
  - SessionSummarizer (id: SessionSummarizer-nxIpBvCSg2)


## Step 3: Test Short-Term Memory\n
<hr>

In [16]:
from datetime import datetime
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
from strands import Agent
from strands.models import BedrockModel

# Test with a simple agent
test_session_id = f"test-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
test_actor_id = "test-user"

config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=test_session_id,
    actor_id=test_actor_id,
)

session_manager = AgentCoreMemorySessionManager(config, region_name=REGION)
model = BedrockModel(model_id="global.anthropic.claude-opus-4-6-v1")
agent = Agent(
    model=model,
    system_prompt="You are a helpful assistant. Be bried in your responses.",
    session_manager=session_manager,
)

response = agent("My name is John and I love football betting.")

Hey John! Nice to meet you. Football betting can be exciting — whether it's NFL, college football, or soccer. 

Are you looking for tips, strategies, help understanding odds, or just want to chat about it? Let me know how I can help! 🏈⚽

In [17]:
response = agent("What sport do I like?")

Based on what you told me, you like **football** — though you didn't specify if that's American football (NFL/college) or soccer (which is called "football" in most of the world). 

Which one is it? 😄

## Step 4: Test Long-Term Memory (Cross-Session)\n

Wait ~30 seconds after Step 3 for the long-term strategies to process, then run this with a **new session ID** but the **same actor ID**.\n
<hr>

In [18]:
# New session, same actor — should recall preferences from long-term memory
new_session_id = f"test-session-2-{datetime.now().strftime('%Y%m%d%H%M%S')}"

config2 = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=new_session_id,
    actor_id=test_actor_id,  # same actor as before
)

session_manager2 = AgentCoreMemorySessionManager(config2, region_name=REGION)
agent2 = Agent(
    model=model,
    system_prompt="You are a helpful assistant. Use everything you know about the user.",
    session_manager=session_manager2,
)

response = agent2("Hi, I'm back. What do you remember about me?")

print("\nLong-term memory test complete!")

Hi, welcome back! I appreciate you reaching out, but I should be straightforward with you — I don't actually have the ability to remember previous conversations. Each conversation I have starts fresh, without any memory of past interactions.

So while I'd love to say I remember you, I'm starting with a clean slate here! But I'm happy to get to know you again. Feel free to share anything you'd like, and we can pick up from wherever you'd like to start. 😊
Long-term memory test complete!


## Step 5: Inspect Memory Records\n
<hr>

In [19]:
# List long-term memory records for the test actor
import boto3
print("--- Long-term memory records ---")
data_client = boto3.client('bedrock-agentcore', region_name=REGION)

# Search across all namespace paths for the test actor
namespace_paths = [
    f"/facts/actor/{test_actor_id}/",
    f"/summaries/actor/{test_actor_id}/",
    f"/preferences/actor/{test_actor_id}/",
]

for ns_path in namespace_paths:
    print(f"\n  Namespace: {ns_path}")
    try:
        records = data_client.list_memory_records(memoryId=memory_id, namespacePath=ns_path)
        for r in records.get("memoryRecords", []):
            print(f"    [{r.get('namespace', 'unknown')}] {r.get('content', {}).get('text', 'N/A')[:100]}")
        if not records.get("memoryRecords"):
            print("    No records yet.")
    except Exception as e:
        print(f"    Error: {e}")

print("\nIf no records yet, long-term extraction may still be processing.")
print("Wait 30-60 seconds and re-run this cell.")

--- Long-term memory records ---

  Namespace: /facts/actor/test-user/
    No records yet.

  Namespace: /summaries/actor/test-user/
    No records yet.

  Namespace: /preferences/actor/test-user/
    No records yet.

If no records yet, long-term extraction may still be processing.
Wait 30-60 seconds and re-run this cell.


## Clean Up (run only when done with the demo)\n
<hr>

In [ ]:
# # Uncomment to delete the memory resource
# client.delete_memory(memory_id=memory_id)
# print(f"Deleted memory: {memory_id}")